This project has been developed and run on Google Colab Notebook

In [ ]:

import os
from langchain_openai import AzureChatOpenAI
from langgraph.graph import MessagesState, StateGraph, START
from langgraph.prebuilt import InjectedState
from langgraph.types import Command, interrupt
from langgraph.checkpoint.memory import MemorySaver
from langchain.tools import BaseTool
from langchain.agents import create_agent
from typing import Any
from openai import AzureOpenAI
from dotenv import load_dotenv

# Load environment variables from a local .env file if available.
load_dotenv()

# Configure Azure OpenAI credentials.
AZURE_ENDPOINT = os.getenv("AZURE_OPENAI_ENDPOINT")
AZURE_API_KEY = os.getenv("AZURE_OPENAI_API_KEY")

client = AzureOpenAI(
    api_key=AZURE_API_KEY,
    api_version="2024-02-15-preview",
    azure_endpoint=AZURE_ENDPOINT,
)

# Set deployment name exactly as mapped in Azure AI Studio.
LLM_MODEL = "gpt-4o-mini"
llm = AzureChatOpenAI(
    azure_deployment=LLM_MODEL,
    api_version="2024-02-15-preview",
    temperature=0.2,
)

class MultiAgentState(MessagesState):
    last_active_agent: str


def get_travel_recommendations(query: str) -> str:
    """Return travel recommendations based on a simple keyword heuristic."""
    query_lower = query.lower()
    if "beach" in query_lower:
        return "For a beach vacation, I recommend Hawaii, Bali, or the Maldives."
    if "mountain" in query_lower:
        return "For a mountain getaway, consider the Swiss Alps, the Rockies, or the Himalayas."
    if "city" in query_lower:
        return "For a city break, I suggest exploring New York, Paris, or Tokyo."
    return "I recommend visiting popular destinations like Hawaii, Bali, or Paris."


def make_handoff_tool(agent_name: str) -> BaseTool:
    class HandoffTool(BaseTool):
        name: str = f"handoff_to_{agent_name}"
        description: str = f"Hand off the conversation to {agent_name}."

        def _run(self, tool_input: str, **kwargs: Any) -> str:
            # Use this tool to indicate the transition to the other agent.
            return f"Switching to {agent_name}"

        async def _arun(self, tool_input: str, **kwargs: Any) -> str:
            return f"Switching to {agent_name}"

    return HandoffTool()


travel_advisor_tools = [
    get_travel_recommendations,
    make_handoff_tool(agent_name="hotel_advisor"),
]

travel_advisor = create_agent(
    model=llm,
    tools=travel_advisor_tools,
    system_prompt=(
        "You are a general travel expert that can recommend travel destinations "
        "(e.g. countries, cities, etc). If you need hotel recommendations, ask 'hotel_advisor' for help. "
        "You MUST include a human-readable response before transferring to another agent."
    ),
)


def call_travel_advisor(state: MultiAgentState) -> Command:
    response = travel_advisor.invoke(state)
    update = {**response, "last_active_agent": "travel_advisor"}
    # After the travel advisor responds, return control to the human node.
    return Command(update=update, goto="human")


def get_hotel_recommendations(query: str) -> str:
    """Return hotel recommendations using a simple keyword heuristic."""
    query_lower = query.lower()
    if "beach" in query_lower:
        return "For beach hotels, I recommend The Ritz-Carlton, Bali or Four Seasons Maui."
    if "city" in query_lower:
        return "For city hotels, consider The Peninsula, Hong Kong or The Savoy, London."
    return "I recommend checking out hotels like The Ritz-Carlton or Four Seasons."


hotel_advisor_tools = [
    get_hotel_recommendations,
    make_handoff_tool(agent_name="travel_advisor"),
]

hotel_advisor = create_agent(
    model=llm,
    tools=hotel_advisor_tools,
    system_prompt=(
        "You are a hotel expert that can provide hotel recommendations for a given destination. "
        "If you need help picking travel destinations, ask 'travel_advisor' for help. "
        "You MUST include a human-readable response before transferring to another agent."
    ),
)


def call_hotel_advisor(state: MultiAgentState) -> Command:
    response = hotel_advisor.invoke(state)
    update = {**response, "last_active_agent": "hotel_advisor"}
    # After the hotel advisor responds, return control to the human node.
    return Command(update=update, goto="human")


def human_node(state: MultiAgentState, config) -> Command:
    # Emit an interrupt so the graph waits for the next human message.
    user_input = interrupt(value="Ready for user input.")
    active_agent = state["last_active_agent"]
    return Command(
        update={"messages": [{"role": "human", "content": user_input}]},
        goto=active_agent,
    )


builder = StateGraph(MultiAgentState)
builder.add_node("travel_advisor", call_travel_advisor)
builder.add_node("hotel_advisor", call_hotel_advisor)
builder.add_node("human", human_node)
# Start the conversation with the travel advisor.
builder.add_edge(START, "travel_advisor")
checkpointer = MemorySaver()
graph = builder.compile(checkpointer=checkpointer)

In [4]:
import uuid

thread_config = {"configurable": {"thread_id": str(uuid.uuid4())}}

# Sample conversation turns for the multi-agent graph.
inputs = [
    {"messages": [{"role": "user", "content": "i wanna go somewhere warm in the caribbean"}]},
    Command(resume="could you recommend a nice hotel in one of the areas and tell me which area it is."),
    Command(resume="i like the first one. could you recommend something to do near the hotel?"),
]

for idx, user_input in enumerate(inputs):
    print(f"\n--- Conversation Turn {idx + 1} ---\n")
    print(f"User: {user_input}\n")
    for update in graph.stream(user_input, config=thread_config, stream_mode="updates"):
        for node_id, value in update.items():
            if isinstance(value, dict) and value.get("messages", []):
                last_message = value["messages"][-1]
                if isinstance(last_message, dict) or last_message.type != "ai":
                    continue
                print(f"{node_id}: {last_message.content}")



--- Conversation Turn 1 ---

User: {'messages': [{'role': 'user', 'content': 'i wanna go somewhere warm in the caribbean'}]}

travel_advisor: It seems there was a mix-up in the recommendations. For warm Caribbean destinations, I suggest considering places like:

1. **Jamaica** - Known for its beautiful beaches, reggae music, and vibrant culture.
2. **Barbados** - Offers stunning coastlines, rich history, and delicious cuisine.
3. **Dominican Republic** - Famous for its all-inclusive resorts and beautiful landscapes.
4. **Cuba** - Known for its rich history, classic cars, and beautiful beaches.
5. **St. Lucia** - Renowned for its stunning natural beauty, including the Pitons and lush rainforests.

If you need hotel recommendations for any of these destinations, just let me know!

--- Conversation Turn 2 ---

User: Command(resume='could you recommend a nice hotel in one of the areas and tell me which area it is.')

travel_advisor: I'm transferring you to a hotel advisor who can help you